In [ ]:
import os
open('/kaggle/working/radar_audit.py','w',encoding='utf-8').write('"""Stage C0: audit the nuScenes-mini Kaggle dataset for RADAR. Pure numpy; no dataset is faked."""\nimport collections, glob, json, os, sys, zipfile\nimport numpy as np\nimport matplotlib\nmatplotlib.use(\'Agg\')\nimport matplotlib.pyplot as plt\n\nOUT = sys.argv[1]\nos.makedirs(OUT, exist_ok=True)\nV = None\nfor r, d, f in os.walk(\'/kaggle/input\'):\n    if \'v1.0-mini\' in d:\n        V = os.path.join(r, \'v1.0-mini\'); break\nROOT = os.path.dirname(V)\nprint(\'ROOT\', ROOT)\nRADARS = [\'RADAR_FRONT\', \'RADAR_FRONT_LEFT\', \'RADAR_FRONT_RIGHT\', \'RADAR_BACK_LEFT\', \'RADAR_BACK_RIGHT\']\nCAMS = [\'CAM_FRONT\', \'CAM_FRONT_LEFT\', \'CAM_FRONT_RIGHT\', \'CAM_BACK\', \'CAM_BACK_LEFT\', \'CAM_BACK_RIGHT\']\n\n\ndef load(n):\n    p = os.path.join(V, n + \'.json\')\n    if os.path.exists(p):\n        return json.load(open(p))\n    with zipfile.ZipFile(p + \'.zip\') as z:\n        return json.load(z.open(z.namelist()[0]))\n\n\nsample = {s[\'token\']: s for s in load(\'sample\')}\nsd = load(\'sample_data\')\ncs = {c[\'token\']: c for c in load(\'calibrated_sensor\')}\nego = {e[\'token\']: e for e in load(\'ego_pose\')}\nsensor = {s[\'token\']: s for s in load(\'sensor\')}\nscene = {s[\'token\']: s for s in load(\'scene\')}\nrep = dict(root=ROOT, n_samples=len(sample), n_scenes=len(scene))\n\n# ---- 1. file counts ------------------------------------------------------------------------------------------------------\ncnt = {}\nfor r in RADARS + [\'LIDAR_TOP\'] + CAMS:\n    ns = len(os.listdir(f\'{ROOT}/samples/{r}\')) if os.path.isdir(f\'{ROOT}/samples/{r}\') else -1\n    nw = len(os.listdir(f\'{ROOT}/sweeps/{r}\')) if os.path.isdir(f\'{ROOT}/sweeps/{r}\') else -1\n    cnt[r] = dict(samples=ns, sweeps=nw)\nrep[\'file_counts\'] = cnt\nprint(\'file counts\'); [print(f\'  {k:18s} samples={v["samples"]:5d} sweeps={v["sweeps"]:6d}\') for k, v in cnt.items()]\n\n# ---- 2. table <-> files ------------------------------------------------------------------------------------------------------\nby_ch = collections.defaultdict(list)\nfor d in sd:\n    by_ch[d[\'filename\'].split(\'/\')[1]].append(d)\nmiss = {}\nfor r in RADARS:\n    miss[r] = dict(key_total=sum(1 for d in by_ch[r] if d[\'is_key_frame\']), sweep_total=sum(1 for d in by_ch[r] if not d[\'is_key_frame\']),\n                   missing=sum(1 for d in by_ch[r] if not os.path.exists(f\'{ROOT}/{d["filename"]}\')))\nrep[\'table_vs_files\'] = miss\nprint(\'table vs files\', json.dumps(miss))\n\n# ---- 3. calibration ------------------------------------------------------------------------------------------------------\ncal = {}\nfor r in RADARS:\n    ds = by_ch[r]\n    c = cs[ds[0][\'calibrated_sensor_token\']]\n    tset = {tuple(np.round(cs[d[\'calibrated_sensor_token\']][\'translation\'], 4)) for d in ds}\n    cal[r] = dict(translation=c[\'translation\'], rotation=c[\'rotation\'], intrinsic=c[\'camera_intrinsic\'],\n                  sensor_modality=sensor[c[\'sensor_token\']][\'modality\'], n_distinct_extrinsics=len(tset))\nrep[\'calibration\'] = cal\nprint(\'radar calibration\'); [print(f\'  {k:18s} t={np.round(v["translation"], 3).tolist()} q={np.round(v["rotation"], 3).tolist()} modality={v["sensor_modality"]} distinct={v["n_distinct_extrinsics"]}\') for k, v in cal.items()]\n\n# ---- 4. timestamps ------------------------------------------------------------------------------------------------------\nkf = collections.defaultdict(dict)\nfor d in sd:\n    if d[\'is_key_frame\']:\n        kf[d[\'sample_token\']][d[\'filename\'].split(\'/\')[1]] = d\ndt = {r: [] for r in RADARS + CAMS}\nfor t, chs in kf.items():\n    tl = chs[\'LIDAR_TOP\'][\'timestamp\']\n    for r in RADARS + CAMS:\n        if r in chs:\n            dt[r].append((chs[r][\'timestamp\'] - tl) / 1e3)  # ms\nrep[\'keyframe_dt_ms_vs_lidar\'] = {r: dict(n=len(v), mean=float(np.mean(v)), std=float(np.std(v)), min=float(np.min(v)), max=float(np.max(v)),\n                                          abs_p95=float(np.percentile(np.abs(v), 95))) for r, v in dt.items() if v}\nprint(\'keyframe dt (ms) vs LIDAR_TOP\'); [print(f\'  {r:18s} n={v["n"]} mean={v["mean"]:+.1f} std={v["std"]:.1f} range=[{v["min"]:+.1f},{v["max"]:+.1f}]\') for r, v in rep[\'keyframe_dt_ms_vs_lidar\'].items()]\nsw = {}\nfor r in RADARS:\n    ts = sorted(d[\'timestamp\'] for d in by_ch[r])\n    g = np.diff(ts) / 1e3\n    g = g[(g > 0) & (g < 500)]\n    sw[r] = dict(median_gap_ms=float(np.median(g)), hz=float(1000 / np.median(g)))\nrep[\'radar_rate\'] = sw\nprint(\'radar rate\', {k: round(v[\'hz\'], 1) for k, v in sw.items()})\n\n# ---- 5. fields ------------------------------------------------------------------------------------------------------\nTY = {(\'F\', 4): \'<f4\', (\'F\', 8): \'<f8\', (\'I\', 1): \'i1\', (\'I\', 2): \'<i2\', (\'I\', 4): \'<i4\', (\'U\', 1): \'u1\', (\'U\', 2): \'<u2\', (\'U\', 4): \'<u4\'}\n\n\ndef read_pcd(p):\n    with open(p, \'rb\') as f:\n        h = {}\n        while True:\n            k, *v = f.readline().decode().strip().split()\n            h[k] = v\n            if k == \'DATA\':\n                break\n        dtp = np.dtype([(n, TY[(t, int(s))]) for n, t, s in zip(h[\'FIELDS\'], h[\'TYPE\'], h[\'SIZE\'])])\n        return h, np.frombuffer(f.read(), dtype=dtp, count=int(h[\'POINTS\'][0]))\n\n\nh0, a0 = read_pcd(f\'{ROOT}/{by_ch["RADAR_FRONT"][0]["filename"]}\')\nrep[\'pcd_header\'] = dict(fields=h0[\'FIELDS\'], types=h0[\'TYPE\'], sizes=h0[\'SIZE\'], data=h0[\'DATA\'], version=h0.get(\'VERSION\'))\nprint(\'PCD fields:\', h0[\'FIELDS\'])\nnpts = {r: [] for r in RADARS}\nallp = collections.defaultdict(list)\nfor r in RADARS:\n    for d in [x for x in by_ch[r] if x[\'is_key_frame\']]:\n        _, a = read_pcd(f\'{ROOT}/{d["filename"]}\')\n        npts[r].append(len(a))\n        if len(allp[r]) < 60:\n            allp[r].append(a)\nrep[\'points_per_keyframe\'] = {r: dict(n_frames=len(v), mean=float(np.mean(v)), min=int(np.min(v)), max=int(np.max(v)), zero_frames=int(np.sum(np.array(v) == 0))) for r, v in npts.items()}\nprint(\'points per keyframe\'); [print(f\'  {r:18s} {v}\') for r, v in rep[\'points_per_keyframe\'].items()]\nA = np.concatenate([np.concatenate(v) for v in allp.values()])\nfs = {}\nfor n in A.dtype.names:\n    x = A[n].astype(float)\n    fs[n] = dict(min=float(x.min()), max=float(x.max()), mean=float(x.mean()), n_unique=int(len(np.unique(x))) if len(np.unique(x)) < 64 else \'>63\')\nrep[\'field_stats\'] = fs\nrep[\'invalid_state_hist\'] = {int(k): int(v) for k, v in zip(*np.unique(A[\'invalid_state\'], return_counts=True))}\nrep[\'dyn_prop_hist\'] = {int(k): int(v) for k, v in zip(*np.unique(A[\'dyn_prop\'], return_counts=True))}\nrep[\'ambig_state_hist\'] = {int(k): int(v) for k, v in zip(*np.unique(A[\'ambig_state\'], return_counts=True))}\nprint(\'field stats\'); [print(f\'  {k:16s} {v}\') for k, v in fs.items()]\nprint(\'invalid_state\', rep[\'invalid_state_hist\'], \'| dyn_prop\', rep[\'dyn_prop_hist\'], \'| ambig\', rep[\'ambig_state_hist\'])\n\n# ---- 6. visual proof: radar + lidar in ego frame for a real sample ----------------------------------------------------------------------\ndef qR(q):\n    w, x, y, z = q\n    return np.array([[1 - 2 * (y * y + z * z), 2 * (x * y - z * w), 2 * (x * z + y * w)],\n                     [2 * (x * y + z * w), 1 - 2 * (x * x + z * z), 2 * (y * z - x * w)],\n                     [2 * (x * z - y * w), 2 * (y * z + x * w), 1 - 2 * (x * x + y * y)]])\n\n\ntok = sorted(kf.keys(), key=lambda t: sample[t][\'timestamp\'])[40]\nfig, ax = plt.subplots(1, 2, figsize=(16, 8))\nlid = kf[tok][\'LIDAR_TOP\']\nL = np.fromfile(f\'{ROOT}/{lid["filename"]}\', dtype=np.float32).reshape(-1, 5)[:, :3]\ncl = cs[lid[\'calibrated_sensor_token\']]\nLe = (qR(cl[\'rotation\']) @ L.T).T + np.array(cl[\'translation\'])\ncols = dict(zip(RADARS, [\'tab:red\', \'tab:orange\', \'tab:green\', \'tab:blue\', \'tab:purple\']))\ntot = 0\nfor a_, title in zip(ax, [\'LiDAR (grey) + 5 radars (colour), ego frame\', \'radar only, arrows = ego-motion-compensated velocity\']):\n    a_.set_aspect(\'equal\'); a_.set_xlim(-60, 60); a_.set_ylim(-60, 60); a_.grid(alpha=.3); a_.set_title(title)\nax[0].scatter(-Le[:, 1], Le[:, 0], s=.2, c=\'0.6\')\nfor r in RADARS:\n    d = kf[tok][r]\n    _, a = read_pcd(f\'{ROOT}/{d["filename"]}\')\n    c = cs[d[\'calibrated_sensor_token\']]\n    R = qR(c[\'rotation\'])\n    P = (R @ np.stack([a[\'x\'], a[\'y\'], a[\'z\']]).astype(float)).T + np.array(c[\'translation\'])\n    Vv = (R @ np.stack([a[\'vx_comp\'], a[\'vy_comp\'], np.zeros(len(a))]).astype(float)).T\n    tot += len(a)\n    ax[0].scatter(-P[:, 1], P[:, 0], s=14, c=cols[r], label=f\'{r} ({len(a)})\')\n    ax[1].scatter(-P[:, 1], P[:, 0], s=14, c=cols[r])\n    ax[1].quiver(-P[:, 1], P[:, 0], -Vv[:, 1], Vv[:, 0], color=cols[r], angles=\'xy\', scale_units=\'xy\', scale=1, width=.002)\nax[0].legend(fontsize=8); ax[0].plot(0, 0, \'k^\')\nfig.suptitle(f\'nuScenes-mini sample {tok[:8]}  |  {tot} radar points over 5 sensors, {len(L)} LiDAR points\')\nfig.savefig(f\'{OUT}/radar_lidar_ego_sample.png\', dpi=90, bbox_inches=\'tight\')\n\n# ---- verdict ------------------------------------------------------------------------------------------------------\nok = all(cnt[r][\'samples\'] == 404 for r in RADARS) and all(v[\'missing\'] == 0 for v in miss.values()) \\\n    and all(v[\'zero_frames\'] < v[\'n_frames\'] * 0.2 for v in rep[\'points_per_keyframe\'].values()) \\\n    and all(v[\'n_distinct_extrinsics\'] >= 1 for v in cal.values()) and \'vx_comp\' in h0[\'FIELDS\']\nrep[\'verdict\'] = \'RADAR DATASET AUDIT PASS\' if ok else \'RADAR DATASET AUDIT FAIL\'\njson.dump(rep, open(f\'{OUT}/radar_audit.json\', \'w\'), indent=2)\nprint(rep[\'verdict\'])\n')
print('written')

In [ ]:
%%bash
set -e
mkdir -p /kaggle/working/radar_audit
python3 /kaggle/working/radar_audit.py /kaggle/working/radar_audit 2>&1 | tail -80